# Scaled Dot-Product Attention in PyTorch

Walkthrough of how the attention mechanism calculates context-aware representations for a simple sentence.

## 1. Setup and Tokenization
We start with a simple sentence: **"I love AI and Machine Learning"**.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
sentence = "I love AI and Machine Learning"
words = sentence.split()
words, sentence

In [ ]:
# Simple vocabulary mapping

vocab = {word: idx for idx, word in enumerate(words)}
indices = torch.tensor([vocab[w] for w in words], dtype=torch.long)

print(f"vocab: {vocab}")
print(f"Token Indices: {indices}")

## 2. Embeddings
Convert tokens into continuous vectors.

In [ ]:
embed_dim = 16  # Dimension of the vector
torch.manual_seed(42) # For reproducibility
embedding = nn.Embedding(num_embeddings=len(vocab), embedding_dim=embed_dim)
embedded = embedding(indices)

print(f"Embedded shape: {embedded.shape}") # (seq_len, embed_dim)

In [ ]:
embedded

## 3. Projection to Q, K, V
Each word generates three vectors: **Query**, **Key**, and **Value**.

In [ ]:
d_k = embed_dim
Q_proj = nn.Linear(embed_dim, d_k, bias=False)
K_proj = nn.Linear(embed_dim, d_k, bias=False)
V_proj = nn.Linear(embed_dim, d_k, bias=False)

queries = Q_proj(embedded)
keys = K_proj(embedded)
values = V_proj(embedded)

print(f"Queries shape: {queries.shape}")
print(f"Keys shape: {keys.shape}")
print(f"Values shape: {values.shape}")

In [ ]:
queries

## 4. Calculating Attention Weights
We calculate similarity scores between Queries and Keys, then apply Softmax.

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

In [ ]:
import math

# 1. Dot product of Q and K
attn_scores = torch.matmul(queries, keys.transpose(-2, -1))

# 2. Scaling to prevent gradient vanishing
attn_scores = attn_scores / math.sqrt(d_k)

# 3. Softmax to get probability distribution (weights sum to 1)
attn_weights = F.softmax(attn_scores, dim=-1)

print("Attention Weights Matrix:")
print(attn_weights)
print(f"Sum of weights for first word: {attn_weights[0].sum().item():.4f}")

## 5. Computing the Context Vector
The final output is the weighted sum of the **Value** vectors.

In [ ]:
context = torch.matmul(attn_weights, values)
print(f"Context Vector shape: {context.shape}")
print(context)
print("Context vector for 'I':\n", context[0])
print("Context vector for 'learning':\n", context[5])

## 6. We have the context vector, now what?

The **Context Vector** is a "context-aware" representation of a word. For example, the vector for "I" now contains mixed information from "love" and "AI" based on importance.

**In a Transformer Encoder Block, the following usually happens:**

1.  **Add & Norm (Residual Connection):** The original input (`embedded`) is added to the output of the attention (`context`) and then normalized. This helps prevent training issues like vanishing gradients.
    *   `output = LayerNorm(embedded + context)`
2.  **Feed-Forward Network (FFN):** Each word vector passes through a simple neural network (usually two linear layers with a ReLU/GELU activation) to extract higher-level features.
    *   `ffn_out = FFN(output)`
3.  **Second Add & Norm:** Another residual connection and normalization step.
    *   `Final_Block_Output = LayerNorm(output + ffn_out)`

This final output then serves as the input for the *next* Transformer layer!

## TASK: Implement the attention mechanism using PyTorch existing functions.